In [1]:
import pandas as pd
import sqlite3

In [2]:
# ==========================================
# 1. DONNÉES BRUTES (STAGING AREA)
# ==========================================
# On simule les données brutes (comme si on venait de lire le PDF/Excel)
raw_data = [
    {"Num_CMD": "SLSD/0001", "Date_CMD": "2024-12-28", "Client": "SARL ABC", "Adresse": "Cité 20 Aout, Alger", "Code_Produit": "LAP.0120", "Produit": "Laptop HP Probook G4", "Qté": 4, "Montant_HT": 500000.00},
    {"Num_CMD": "SLSD/0001", "Date_CMD": "2024-12-28", "Client": "SARL ABC", "Adresse": "Cité 20 Aout, Alger", "Code_Produit": "PRI.0020", "Produit": "Printer Canon 6030", "Qté": 1, "Montant_HT": 49000.00},
    {"Num_CMD": "SLSD/0001", "Date_CMD": "2024-12-28", "Client": "SARL ABC", "Adresse": "Cité 20 Aout, Alger", "Code_Produit": "INK.0034", "Produit": "Toner Canon 6030", "Qté": 1, "Montant_HT": 1800.00},
    {"Num_CMD": "SLSR/0002", "Date_CMD": "2025-02-22", "Client": "EURL XYZ", "Adresse": "Coopérative Rym, Blida", "Code_Produit": "LAP.0011", "Produit": "Laptop Lenovo 110", "Qté": 1, "Montant_HT": 89000.00},
    {"Num_CMD": "SLSR/0002", "Date_CMD": "2025-02-22", "Client": "EURL XYZ", "Adresse": "Coopérative Rym, Blida", "Code_Produit": "PRI.0020", "Produit": "Printer Canon 6030", "Qté": 2, "Montant_HT": 98000.00},
    {"Num_CMD": "SLSR/0002", "Date_CMD": "2025-02-22", "Client": "EURL XYZ", "Adresse": "Coopérative Rym, Blida", "Code_Produit": "INK.0004", "Produit": "Toner Canon 6030", "Qté": 2, "Montant_HT": 3600.00},
    {"Num_CMD": "SLSR/0002", "Date_CMD": "2025-02-22", "Client": "EURL XYZ", "Adresse": "Coopérative Rym, Blida", "Code_Produit": "SCA.0002", "Produit": "Scanner Epson 1600", "Qté": 1, "Montant_HT": 21000.00},
    {"Num_CMD": "SLSD/0003", "Date_CMD": "2025-03-15", "Client": "SARL AGRODZ", "Adresse": "Cité 310 logt Kouba, Alger", "Code_Produit": "PRI.0011", "Produit": "Printer EPSON 3010", "Qté": 2, "Montant_HT": 64000.00},
    {"Num_CMD": "SLSD/0003", "Date_CMD": "2025-03-15", "Client": "SARL AGRODZ", "Adresse": "Cité 310 logt Kouba, Alger", "Code_Produit": "LAP.0120", "Produit": "Laptop HP Probook G4", "Qté": 1, "Montant_HT": 125000.00},
    {"Num_CMD": "SLSG/0004", "Date_CMD": "2025-03-28", "Client": "SNC Wiffak", "Adresse": "Boulvard Nord, Sétif", "Code_Produit": "INK.0001", "Produit": "INK Canon 3210", "Qté": 10, "Montant_HT": 18000.00},
    {"Num_CMD": "SLSD/0005", "Date_CMD": "2025-03-28", "Client": "EURL XYZ", "Adresse": "Coopérative Rym, Oran", "Code_Produit": "LAP.0011", "Produit": "Laptop Lenovo 110", "Qté": 3, "Montant_HT": 267000.00},
    {"Num_CMD": "SLSD/0005", "Date_CMD": "2025-03-28", "Client": "EURL XYZ", "Adresse": "Coopérative Rym, Oran", "Code_Produit": "PRI.0011", "Produit": "Printer EPSON 3010", "Qté": 1, "Montant_HT": 32000.00},
    {"Num_CMD": "SLSD/0005", "Date_CMD": "2025-03-28", "Client": "EURL XYZ", "Adresse": "Coopérative Rym, Oran", "Code_Produit": "INK.0005", "Produit": "INK Epson 110", "Qté": 10, "Montant_HT": 8000.00},
    {"Num_CMD": "SLSG/0006", "Date_CMD": "2025-05-02", "Client": "SARL ABC", "Adresse": "Cité 20 Aout, Alger", "Code_Produit": "LAP.0120", "Produit": "Laptop HP Probook G4", "Qté": 2, "Montant_HT": 250000.00},
    {"Num_CMD": "SLSD/0007", "Date_CMD": "2025-05-04", "Client": "EURL HAMIDI", "Adresse": "Promotion Bahia, Oran", "Code_Produit": "PRI.0020", "Produit": "Printer Canon 6030", "Qté": 2, "Montant_HT": 98000.00}
]
df_raw = pd.DataFrame(raw_data)

# Pré-traitement nécessaire pour créer les dimensions
df_raw['Date_CMD'] = pd.to_datetime(df_raw['Date_CMD'])
df_raw['Wilaya'] = df_raw['Adresse'].apply(lambda x: x.split(',')[-1].strip())
df_raw['Forme_Juridique'] = df_raw['Client'].apply(lambda x: x.split(' ')[0])
df_raw['Categorie'] = df_raw['Code_Produit'].apply(lambda x: x.split('.')[0])
df_raw['Type_Vente'] = df_raw['Num_CMD'].apply(lambda x: x.split('/')[0])

In [3]:
# ==========================================
# 2. CRÉATION DES DIMENSIONS (Code ETL)
# ==========================================

# --- DIM CLIENT ---
# On récupère les clients uniques et on leur donne un ID
dim_client = df_raw[['Client', 'Wilaya', 'Forme_Juridique']].drop_duplicates().reset_index(drop=True)
dim_client['ID_Client'] = dim_client.index + 1 # Création d'une clé primaire (1, 2, 3...)

# --- DIM PRODUIT ---
dim_produit = df_raw[['Code_Produit', 'Produit', 'Categorie']].drop_duplicates().reset_index(drop=True)
dim_produit['ID_Produit'] = dim_produit.index + 1

# --- DIM TEMPS ---
dim_temps = df_raw[['Date_CMD']].drop_duplicates().sort_values('Date_CMD').reset_index(drop=True)
dim_temps['ID_Temps'] = dim_temps.index + 1
dim_temps['Annee'] = dim_temps['Date_CMD'].dt.year
dim_temps['Mois'] = dim_temps['Date_CMD'].dt.month
dim_temps['Nom_Mois'] = dim_temps['Date_CMD'].dt.month_name()

# --- DIM TYPE VENTE ---
dim_type = df_raw[['Type_Vente']].drop_duplicates().reset_index(drop=True)
dim_type['ID_Type'] = dim_type.index + 1


In [4]:

# ==========================================
# 3. CRÉATION DE LA TABLE DE FAITS
# ==========================================
# On remplace les textes par les IDs (Clés Étrangères)
fait_ventes = df_raw.copy()

# Jointure pour récupérer ID_Client
fait_ventes = fait_ventes.merge(dim_client, on=['Client', 'Wilaya', 'Forme_Juridique'])
# Jointure pour récupérer ID_Produit
fait_ventes = fait_ventes.merge(dim_produit, on=['Code_Produit', 'Produit', 'Categorie'])
# Jointure pour récupérer ID_Temps
fait_ventes = fait_ventes.merge(dim_temps[['Date_CMD', 'ID_Temps']], on='Date_CMD')
# Jointure pour récupérer ID_Type
fait_ventes = fait_ventes.merge(dim_type, on='Type_Vente')

# On ne garde que les clés (IDs) et les mesures (Qté, Montant)
colonnes_fait = ['ID_Client', 'ID_Produit', 'ID_Temps', 'ID_Type', 'Qté', 'Montant_HT']
fait_ventes_final = fait_ventes[colonnes_fait]


In [5]:

# ==========================================
# 4. CHARGEMENT DANS LE WAREHOUSE (SQLite)
# ==========================================
conn = sqlite3.connect('warehouse_ventes.db')

# On écrit les tables séparément
dim_client.to_sql('Dim_Client', conn, if_exists='replace', index=False)
dim_produit.to_sql('Dim_Produit', conn, if_exists='replace', index=False)
dim_temps.to_sql('Dim_Temps', conn, if_exists='replace', index=False)
dim_type.to_sql('Dim_Type', conn, if_exists='replace', index=False)
fait_ventes_final.to_sql('Fait_Ventes', conn, if_exists='replace', index=False)

print("Warehouse créé avec succès !")
print("\n--- Aperçu de la Table de Faits (Ce qui est stocké) ---")
print(fait_ventes_final.head())
print("\n--- Aperçu d'une Dimension (Dim_Client) ---")
print(dim_client.head())

conn.close()

Warehouse créé avec succès !

--- Aperçu de la Table de Faits (Ce qui est stocké) ---
   ID_Client  ID_Produit  ID_Temps  ID_Type  Qté  Montant_HT
0          1           1         1        1    4    500000.0
1          1           2         1        1    1     49000.0
2          1           3         1        1    1      1800.0
3          2           4         2        2    1     89000.0
4          2           2         2        2    2     98000.0

--- Aperçu d'une Dimension (Dim_Client) ---
        Client Wilaya Forme_Juridique  ID_Client
0     SARL ABC  Alger            SARL          1
1     EURL XYZ  Blida            EURL          2
2  SARL AGRODZ  Alger            SARL          3
3   SNC Wiffak  Sétif             SNC          4
4     EURL XYZ   Oran            EURL          5


In [6]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.io as pio

# --- MODIFICATION POUR RÉGLER L'ERREUR D'AFFICHAGE ---
# Cela va ouvrir les graphiques dans votre navigateur web au lieu de VS Code
pio.renderers.default = "browser"
# -----------------------------------------------------

# ==========================================
# 5. CONNEXION AU WAREHOUSE ET RÉCUPÉRATION (OLAP)
# ==========================================
print(">>> Connexion au Data Warehouse pour analyse...")
conn = sqlite3.connect('warehouse_ventes.db')

# Requête SQL pour récupérer les données avec les noms (Jointures)
query_analytique = """
SELECT 
    t.Date_CMD,
    t.Annee,
    t.Mois,
    t.Nom_Mois,
    p.Produit,
    p.Categorie,
    c.Client,
    c.Wilaya,
    c.Forme_Juridique,
    typ.Type_Vente,
    f.Qté,
    f.Montant_HT
FROM Fait_Ventes f
JOIN Dim_Temps t ON f.ID_Temps = t.ID_Temps
JOIN Dim_Produit p ON f.ID_Produit = p.ID_Produit
JOIN Dim_Client c ON f.ID_Client = c.ID_Client
JOIN Dim_Type typ ON f.ID_Type = typ.ID_Type
"""

# Chargement des résultats dans un DataFrame Pandas
df_analyse = pd.read_sql_query(query_analytique, conn)
conn.close()

# Conversion de la date
df_analyse['Date_CMD'] = pd.to_datetime(df_analyse['Date_CMD'])

print(f"Données récupérées : {len(df_analyse)} lignes chargées.\n")


# ==========================================
# 6. RÉPONSES AUX QUESTIONS D'ANALYSE
# ==========================================

# --- Q1 : Liste des produits vendus après le 01 Février 2025 ---
print("--- Q1 : Produits vendus après le 01/02/2025 ---")
res_q1 = df_analyse[df_analyse['Date_CMD'] > '2025-02-01']
print(res_q1[['Date_CMD', 'Produit', 'Montant_HT']].sort_values('Date_CMD').to_string(index=False))
print("\n" + "="*40 + "\n")


# --- Q2 : Classement produits par CA, Type Vente et Année (Graphique) ---
print("--- Q2 : Génération Graphique (Produits par CA/Type/Année) ---")
df_q2 = df_analyse.groupby(['Produit', 'Type_Vente', 'Annee'])['Montant_HT'].sum().reset_index()
df_q2 = df_q2.sort_values('Montant_HT', ascending=False)

fig2 = px.bar(
    df_q2,
    x='Produit',
    y='Montant_HT',
    color='Type_Vente',
    facet_col='Annee',
    title="Classement des Produits par CA (Segmenté par Année et Type)",
    text_auto='.2s'
)
fig2.show() # S'ouvrira dans le navigateur


# --- Q3 : Classement Clients par Wilaya et Forme Juridique (Graphique) ---
print("--- Q3 : Génération Graphique (Clients par Wilaya/Forme) ---")
df_q3 = df_analyse.groupby(['Wilaya', 'Forme_Juridique', 'Client'])['Montant_HT'].sum().reset_index()

fig3 = px.sunburst(
    df_q3,
    path=['Wilaya', 'Forme_Juridique', 'Client'],
    values='Montant_HT',
    title="Analyse des Ventes par Wilaya et Typologie Client",
    color='Montant_HT',
    color_continuous_scale='RdBu'
)
fig3.show() # S'ouvrira dans le navigateur


# --- Q4 : Ventes quantitatives par Catégorie, Type, Mois et Année ---
print("--- Q4 : Génération Graphique (Quantités par Categorie/Temps) ---")
df_q4 = df_analyse.groupby(['Annee', 'Nom_Mois', 'Categorie', 'Type_Vente'])['Qté'].sum().reset_index()

fig4 = px.bar(
    df_q4,
    x='Nom_Mois',
    y='Qté',
    color='Categorie',
    facet_row='Annee',
    facet_col='Type_Vente',
    title="Évolution Quantitative des Ventes (Multidimensionnelle)",
    barmode='group'
)
fig4.show() # S'ouvrira dans le navigateur


# --- Q5 : Catégorie la plus rentable (Top 1) ---
print("--- Q5 : Catégorie Championne (Rentabilité) ---")
df_q5 = df_analyse.groupby('Categorie')['Montant_HT'].sum().reset_index()
top_cat = df_q5.sort_values('Montant_HT', ascending=False).iloc[0]

print(f" La catégorie la plus rentable est : {top_cat['Categorie']}")
print(f" Chiffre d'Affaires total : {top_cat['Montant_HT']:,.2f} DA")

>>> Connexion au Data Warehouse pour analyse...
Données récupérées : 15 lignes chargées.

--- Q1 : Produits vendus après le 01/02/2025 ---
  Date_CMD              Produit  Montant_HT
2025-02-22    Laptop Lenovo 110     89000.0
2025-02-22   Printer Canon 6030     98000.0
2025-02-22     Toner Canon 6030      3600.0
2025-02-22   Scanner Epson 1600     21000.0
2025-03-15   Printer EPSON 3010     64000.0
2025-03-15 Laptop HP Probook G4    125000.0
2025-03-28       INK Canon 3210     18000.0
2025-03-28    Laptop Lenovo 110    267000.0
2025-03-28   Printer EPSON 3010     32000.0
2025-03-28        INK Epson 110      8000.0
2025-05-02 Laptop HP Probook G4    250000.0
2025-05-04   Printer Canon 6030     98000.0


--- Q2 : Génération Graphique (Produits par CA/Type/Année) ---
Ouverture dans une session de navigateur existante.
--- Q3 : Génération Graphique (Clients par Wilaya/Forme) ---
Ouverture dans une session de navigateur existante.
--- Q4 : Génération Graphique (Quantités par Categorie/Temp

# ============================================================
# PARTIE 2 — MODÉLISATION ET STOCKAGE DES DONNÉES D'ACHATS
# ============================================================

In [9]:
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output

pio.renderers.default = "browser"

# ==========================================
# 1. STAGING AREA — Lecture du CSV brut
# ==========================================
df_achats_raw = pd.read_csv('data/achats.csv')

# Nettoyage des noms de colonnes (espaces, points)
df_achats_raw.columns = [c.strip().replace(' ', '_').replace('.', '_') for c in df_achats_raw.columns]
# Renomme pour cohérence
df_achats_raw.rename(columns={'QTY': 'Qté', 'Montant_HT': 'Montant_HT',
                               'Montant_TTC': 'Montant_TTC', 'Code_Produit': 'Code_Produit'}, inplace=True)

# Colonnes dérivées
df_achats_raw['Date_CMD']      = pd.to_datetime(df_achats_raw['Date_CMD'])
df_achats_raw['Categorie']     = df_achats_raw['Code_Produit'].apply(lambda x: x.split('.')[0])
df_achats_raw['Type_Achat']    = df_achats_raw['Num_CMD'].apply(lambda x: x.split('/')[0])

print("=== Aperçu des données brutes d'achats ===")
print(df_achats_raw.to_string(index=False))
print(f"\nShape : {df_achats_raw.shape[0]} lignes × {df_achats_raw.shape[1]} colonnes")

=== Aperçu des données brutes d'achats ===
 Num_CMD   Date_CMD          Fournisseur Code_Produit              Produit  Qté  Montant_HT      Taxe  Montant_TTC Categorie Type_Achat
POL/0001 2024-11-05 SARL IMPORT COMPUTER     LAP.0120 Laptop HP Probook G4   10   1000000.0  190000.0    1190000.0       LAP        POL
POL/0001 2024-11-05 SARL IMPORT COMPUTER     PRI.0020   Printer Canon 6030   10    390000.0   74100.0     464100.0       PRI        POL
POL/0001 2024-11-05 SARL IMPORT COMPUTER     INK.0034     Toner Canon 6030   20       900.0     171.0       1071.0       INK        POL
POI/0002 2024-12-16             EURL ABM     LAP.0011    Laptop Lenovo 110  500  33500000.0 6365000.0   39865000.0       LAP        POI
POI/0002 2024-12-16             EURL ABM     PRI.0011   Printer EPSON 3010  500  11500000.0 2185000.0   13685000.0       PRI        POI
POI/0002 2024-12-16             EURL ABM     INK.0001       INK Canon 3210 1000    600000.0  114000.0     714000.0       INK        POI
POI/0

In [10]:
# ==========================================
# 2. ETL — CRÉATION DES DIMENSIONS
# ==========================================

# --- DIM FOURNISSEUR ---
dim_fournisseur = (df_achats_raw[['Fournisseur']]
                   .drop_duplicates()
                   .reset_index(drop=True))
dim_fournisseur['ID_Fournisseur'] = dim_fournisseur.index + 1

# --- DIM PRODUIT ACHAT ---
dim_produit_achat = (df_achats_raw[['Code_Produit', 'Produit', 'Categorie']]
                     .drop_duplicates()
                     .reset_index(drop=True))
dim_produit_achat['ID_Produit'] = dim_produit_achat.index + 1

# --- DIM TEMPS ACHAT ---
dim_temps_achat = (df_achats_raw[['Date_CMD']]
                   .drop_duplicates()
                   .sort_values('Date_CMD')
                   .reset_index(drop=True))
dim_temps_achat['ID_Temps']  = dim_temps_achat.index + 1
dim_temps_achat['Annee']     = dim_temps_achat['Date_CMD'].dt.year
dim_temps_achat['Mois']      = dim_temps_achat['Date_CMD'].dt.month
dim_temps_achat['Nom_Mois']  = dim_temps_achat['Date_CMD'].dt.month_name()

# --- DIM TYPE ACHAT ---
dim_type_achat = (df_achats_raw[['Type_Achat']]
                  .drop_duplicates()
                  .reset_index(drop=True))
dim_type_achat['ID_Type'] = dim_type_achat.index + 1

print("Dim_Fournisseur :\n", dim_fournisseur.to_string(index=False))
print("\nDim_Produit_Achat :\n", dim_produit_achat.to_string(index=False))
print("\nDim_Temps_Achat :\n", dim_temps_achat.to_string(index=False))
print("\nDim_Type_Achat :\n", dim_type_achat.to_string(index=False))

Dim_Fournisseur :
          Fournisseur  ID_Fournisseur
SARL IMPORT COMPUTER               1
            EURL ABM               2
          SNC Wiffak               3

Dim_Produit_Achat :
 Code_Produit              Produit Categorie  ID_Produit
    LAP.0120 Laptop HP Probook G4       LAP           1
    PRI.0020   Printer Canon 6030       PRI           2
    INK.0034     Toner Canon 6030       INK           3
    LAP.0011    Laptop Lenovo 110       LAP           4
    PRI.0011   Printer EPSON 3010       PRI           5
    INK.0001       INK Canon 3210       INK           6
    SCA.0002   Scanner Epson 1600       SCA           7
    INK.0005        INK Epson 110       INK           8

Dim_Temps_Achat :
   Date_CMD  ID_Temps  Annee  Mois Nom_Mois
2024-11-05         1   2024    11 November
2024-12-16         2   2024    12 December
2025-02-11         3   2025     2 February
2025-02-25         4   2025     2 February

Dim_Type_Achat :
 Type_Achat  ID_Type
       POL        1
       POI   

In [11]:
# ==========================================
# 3. ETL — CRÉATION DE LA TABLE DE FAITS
# ==========================================
fait_achats = df_achats_raw.copy()

fait_achats = fait_achats.merge(dim_fournisseur,  on='Fournisseur')
fait_achats = fait_achats.merge(dim_produit_achat, on=['Code_Produit', 'Produit', 'Categorie'])
fait_achats = fait_achats.merge(dim_temps_achat[['Date_CMD', 'ID_Temps']], on='Date_CMD')
fait_achats = fait_achats.merge(dim_type_achat,   on='Type_Achat')

colonnes_fait = ['ID_Fournisseur', 'ID_Produit', 'ID_Temps', 'ID_Type',
                 'Qté', 'Montant_HT', 'Taxe', 'Montant_TTC']
fait_achats_final = fait_achats[colonnes_fait].reset_index(drop=True)

print("=== Table de Faits — Fait_Achats ===")
print(fait_achats_final.to_string(index=False))

=== Table de Faits — Fait_Achats ===
 ID_Fournisseur  ID_Produit  ID_Temps  ID_Type  Qté  Montant_HT      Taxe  Montant_TTC
              1           1         1        1   10   1000000.0  190000.0    1190000.0
              1           2         1        1   10    390000.0   74100.0     464100.0
              1           3         1        1   20       900.0     171.0       1071.0
              2           4         2        2  500  33500000.0 6365000.0   39865000.0
              2           5         2        2  500  11500000.0 2185000.0   13685000.0
              2           6         2        2 1000    600000.0  114000.0     714000.0
              2           7         2        2  200   3000000.0  570000.0    3570000.0
              1           1         3        1    5    525000.0   99750.0     624750.0
              1           2         3        1    3    123000.0   23370.0     146370.0
              3           8         4        2 1000    600000.0  114000.0     714000.0


In [12]:
# ==========================================
# 4. CHARGEMENT DANS LE WAREHOUSE (SQLite)
# ==========================================
conn_a = sqlite3.connect('warehouse_achats.db')

dim_fournisseur.to_sql('Dim_Fournisseur',   conn_a, if_exists='replace', index=False)
dim_produit_achat.to_sql('Dim_Produit',     conn_a, if_exists='replace', index=False)
dim_temps_achat.to_sql('Dim_Temps',         conn_a, if_exists='replace', index=False)
dim_type_achat.to_sql('Dim_Type',           conn_a, if_exists='replace', index=False)
fait_achats_final.to_sql('Fait_Achats',     conn_a, if_exists='replace', index=False)

conn_a.close()
print("✔  Warehouse 'warehouse_achats.db' créé avec succès !")

✔  Warehouse 'warehouse_achats.db' créé avec succès !


In [13]:
# ==========================================
# 5. CONNEXION AU WAREHOUSE — OLAP
# ==========================================
conn_a = sqlite3.connect('warehouse_achats.db')

query_achats = """
SELECT
    t.Date_CMD,
    t.Annee,
    t.Mois,
    t.Nom_Mois,
    p.Produit,
    p.Code_Produit,
    p.Categorie,
    f.Fournisseur,
    typ.Type_Achat,
    a.Qté,
    a.Montant_HT,
    a.Taxe,
    a.Montant_TTC
FROM Fait_Achats a
JOIN Dim_Temps      t   ON a.ID_Temps       = t.ID_Temps
JOIN Dim_Produit    p   ON a.ID_Produit      = p.ID_Produit
JOIN Dim_Fournisseur f  ON a.ID_Fournisseur  = f.ID_Fournisseur
JOIN Dim_Type       typ ON a.ID_Type         = typ.ID_Type
"""

df_a = pd.read_sql_query(query_achats, conn_a)
conn_a.close()
df_a['Date_CMD'] = pd.to_datetime(df_a['Date_CMD'])

print(f"✔  {len(df_a)} lignes chargées depuis le warehouse.\n")
print(df_a.to_string(index=False))

✔  10 lignes chargées depuis le warehouse.

  Date_CMD  Annee  Mois Nom_Mois              Produit Code_Produit Categorie          Fournisseur Type_Achat  Qté  Montant_HT      Taxe  Montant_TTC
2024-11-05   2024    11 November Laptop HP Probook G4     LAP.0120       LAP SARL IMPORT COMPUTER        POL   10   1000000.0  190000.0    1190000.0
2024-11-05   2024    11 November   Printer Canon 6030     PRI.0020       PRI SARL IMPORT COMPUTER        POL   10    390000.0   74100.0     464100.0
2024-11-05   2024    11 November     Toner Canon 6030     INK.0034       INK SARL IMPORT COMPUTER        POL   20       900.0     171.0       1071.0
2024-12-16   2024    12 December    Laptop Lenovo 110     LAP.0011       LAP             EURL ABM        POI  500  33500000.0 6365000.0   39865000.0
2024-12-16   2024    12 December   Printer EPSON 3010     PRI.0011       PRI             EURL ABM        POI  500  11500000.0 2185000.0   13685000.0
2024-12-16   2024    12 December       INK Canon 3210     INK.

In [14]:
# ============================================================
# 6. ANALYSE DYNAMIQUE INTERACTIVE  (ipywidgets)
#    L'utilisateur choisit librement les axes et l'indicateur
# ============================================================

AXES       = ['Produit', 'Categorie', 'Fournisseur', 'Type_Achat', 'Nom_Mois', 'Annee']
INDICATEURS = {'Coût d\'achat HT (DA)': 'Montant_HT',
               'Coût d\'achat TTC (DA)': 'Montant_TTC',
               'Quantités achetées':     'Qté'}

w_x    = widgets.Dropdown(options=AXES,        value='Produit',   description='Axe X :')
w_col  = widgets.Dropdown(options=[None]+AXES, value='Categorie', description='Couleur :')
w_facet= widgets.Dropdown(options=[None]+AXES, value=None,        description='Facette :')
w_ind  = widgets.Dropdown(options=list(INDICATEURS.keys()),
                           value='Coût d\'achat HT (DA)',          description='Indicateur :')
w_btn  = widgets.Button(description='Générer le graphe',
                         button_style='success', icon='bar-chart')
w_out  = widgets.Output()

def on_click(_):
    with w_out:
        clear_output(wait=True)
        x_col   = w_x.value
        col_col = w_col.value if w_col.value else None
        fac_col = w_facet.value if w_facet.value else None
        ind_col = INDICATEURS[w_ind.value]

        grp = [x_col]
        if col_col and col_col != x_col: grp.append(col_col)
        if fac_col and fac_col not in grp: grp.append(fac_col)

        df_plot = df_a.groupby(grp)[ind_col].sum().reset_index()

        fig = px.bar(
            df_plot,
            x=x_col,
            y=ind_col,
            color=col_col,
            facet_col=fac_col,
            title=f"{w_ind.value} par {x_col}"
                  + (f" / {col_col}" if col_col else "")
                  + (f" — Facette : {fac_col}" if fac_col else ""),
            text_auto='.2s',
            barmode='group'
        )
        fig.update_layout(xaxis_tickangle=-30)
        fig.show()

w_btn.on_click(on_click)

ui = widgets.VBox([
    widgets.HTML("<h3>🔎 Analyse dynamique des achats</h3>"),
    widgets.HBox([w_x, w_col, w_facet, w_ind]),
    w_btn,
    w_out
])
display(ui)

In [ ]:
# ============================================================
# 7. QUESTIONS D'ANALYSE
# ============================================================

# ── Q1 : Liste des produits achetés en 2024 ─────────────────
print("─" * 55)
print("Q1 — Produits achetés en 2024")
print("─" * 55)
res_q1 = (df_a[df_a['Annee'] == 2024][['Date_CMD', 'Produit', 'Categorie', 'Fournisseur', 'Qté', 'Montant_HT']]
          .sort_values('Date_CMD')
          .drop_duplicates())
print(res_q1.to_string(index=False))
print()

# ── Q2 : Achats quantitatifs par produit / type / mois / année (graphe) ──
print("─" * 55)
print("Q2 — Achats quantitatifs (Produit × Type × Mois × Année)")
print("─" * 55)
df_q2 = df_a.groupby(['Annee', 'Nom_Mois', 'Mois', 'Produit', 'Type_Achat'])['Qté'].sum().reset_index()
df_q2 = df_q2.sort_values(['Annee', 'Mois'])

fig_q2 = px.bar(
    df_q2,
    x='Produit',
    y='Qté',
    color='Type_Achat',
    facet_col='Nom_Mois',
    facet_row='Annee',
    title="Achats quantitatifs par Produit, Type d'achat, Mois et Année",
    text_auto=True,
    barmode='group'
)
fig_q2.update_layout(xaxis_tickangle=-45)
fig_q2.show()

# ── Q3 : Fournisseur avec le plus d'achats par catégorie (graphe) ────────
print("─" * 55)
print("Q3 — Fournisseur champion par Catégorie produit")
print("─" * 55)
df_q3 = df_a.groupby(['Categorie', 'Fournisseur'])['Montant_HT'].sum().reset_index()
df_q3 = df_q3.sort_values(['Categorie', 'Montant_HT'], ascending=[True, False])

# Top fournisseur par catégorie (tableau)
top_fourn = df_q3.loc[df_q3.groupby('Categorie')['Montant_HT'].idxmax()]
print(top_fourn.to_string(index=False))
print()

fig_q3 = px.bar(
    df_q3,
    x='Categorie',
    y='Montant_HT',
    color='Fournisseur',
    title="Volume d'achats (HT) par Catégorie et Fournisseur",
    text_auto='.2s',
    barmode='group'
)
fig_q3.show()

# Sunburst complémentaire
fig_q3b = px.sunburst(
    df_q3,
    path=['Categorie', 'Fournisseur'],
    values='Montant_HT',
    title="Répartition des achats — Catégorie › Fournisseur",
    color='Montant_HT',
    color_continuous_scale='Blues'
)
fig_q3b.show()

# ── Q4 : Catégorie ayant coûté le plus ──────────────────────
print("─" * 55)
print("Q4 — Catégorie la plus coûteuse")
print("─" * 55)
df_q4 = df_a.groupby('Categorie')['Montant_HT'].sum().reset_index()
df_q4 = df_q4.sort_values('Montant_HT', ascending=False)
top = df_q4.iloc[0]
print(df_q4.to_string(index=False))
print(f"\n  ➤ La catégorie la plus coûteuse : {top['Categorie']}")
print(f"    Coût total HT : {top['Montant_HT']:>15,.2f} DA")

fig_q4 = px.pie(
    df_q4,
    names='Categorie',
    values='Montant_HT',
    title="Répartition du coût d'achat par Catégorie de produit",
    hole=0.4
)
fig_q4.show()